In [0]:
# =====================================================================
# VALIDACIÓN DE CAPA SILVER EN S3
# Recorre todas las clases en silver/ e imprime:
#   - Cantidad de archivos por clase (.npz y .csv)
#   - Tamaño total por clase (MB)
#   - Total de beats cargados por clase (leyendo los .npz)
#   - Resumen general
# Ejecutar en Databricks.
# =====================================================================
import numpy as np
import io
from rich import print
from rich.table import Table

BUCKET_OUTPUT = 'shazam-proyecto-ecg'
S3_BASE = f"s3://{BUCKET_OUTPUT}/silver"

CLASES = {
    "clase_0_normal":  "Normal (Sinus)",
    "clase_1_afib":    "AFib",
    "clase_2_pvc":     "PVC",
    "clase_3_lbbb":    "LBBB",
}

print("[bold cyan]═══════════════════════════════════════════════[/bold cyan]")
print("[bold cyan]   VALIDACIÓN DE CAPA SILVER EN S3[/bold cyan]")
print("[bold cyan]═══════════════════════════════════════════════[/bold cyan]")
print(f"[white]Bucket : {BUCKET_OUTPUT}[/white]")
print(f"[white]Prefijo: silver/[/white]\n")

# ── Recolectar info por clase ───────────────────────────────────────
resumen = []

for carpeta, label in CLASES.items():
    ruta = f"{S3_BASE}/{carpeta}/"
    try:
        archivos = dbutils.fs.ls(ruta)
    except Exception:
        resumen.append({
            "clase": carpeta, "label": label,
            "n_archivos": 0, "n_npz": 0, "n_csv": 0,
            "size_bytes": 0, "total_beats": 0, "leads": 0,
            "beat_length": 0, "error": True,
        })
        continue

    n_npz = 0
    n_csv = 0
    n_otros = 0
    size_total = 0
    total_beats = 0
    leads = 0
    beat_length = 0

    for f in archivos:
        size_total += f.size
        nombre = f.name.lower()
        if nombre.endswith(".npz"):
            n_npz += 1
            # Leer via Spark binaryFile (usa el conector S3 existente)
            try:
                row = spark.read.format("binaryFile").load(f.path).first()
                buf = io.BytesIO(row.content)
                data = np.load(buf)
                arr = data[list(data.keys())[0]]
                total_beats += arr.shape[0]
                if leads == 0 and arr.ndim == 3:
                    leads = arr.shape[1]
                    beat_length = arr.shape[2]
                data.close()
                buf.close()
            except Exception as e:
                print(f"[yellow]  ⚠ No se pudo leer {nombre}: {e}[/yellow]")
        elif nombre.endswith(".csv"):
            n_csv += 1
        else:
            n_otros += 1

    resumen.append({
        "clase": carpeta, "label": label,
        "n_archivos": len(archivos), "n_npz": n_npz, "n_csv": n_csv,
        "size_bytes": size_total, "total_beats": total_beats,
        "leads": leads, "beat_length": beat_length, "error": False,
    })

# ── Tabla de resultados ─────────────────────────────────────────────
tabla = Table(title="Resumen Capa Silver", show_lines=True)
tabla.add_column("Clase", style="bold white")
tabla.add_column("Label", style="cyan")
tabla.add_column("Archivos", justify="right")
tabla.add_column("NPZ", justify="right", style="green")
tabla.add_column("CSV", justify="right", style="green")
tabla.add_column("Tamaño (MB)", justify="right", style="yellow")
tabla.add_column("Total Beats", justify="right", style="bold magenta")
tabla.add_column("Shape beat", justify="center")

total_beats_global = 0
total_size_global = 0
total_archivos_global = 0

for r in resumen:
    if r["error"]:
        tabla.add_row(
            r["clase"], r["label"],
            "[red]NO EXISTE[/red]", "-", "-", "-", "-", "-"
        )
        continue

    size_mb = r["size_bytes"] / (1024 * 1024)
    shape_str = f"({r['leads']}, {r['beat_length']})" if r["leads"] > 0 else "-"

    tabla.add_row(
        r["clase"], r["label"],
        str(r["n_archivos"]),
        str(r["n_npz"]),
        str(r["n_csv"]),
        f"{size_mb:,.1f}",
        f"{r['total_beats']:,}",
        shape_str,
    )

    total_beats_global += r["total_beats"]
    total_size_global += r["size_bytes"]
    total_archivos_global += r["n_archivos"]

print(tabla)

# ── Totales globales ────────────────────────────────────────────────
print(f"\n[bold white]{'─' * 50}[/bold white]")
print(f"[bold white]TOTALES GLOBALES[/bold white]")
print(f"  Archivos totales : [green]{total_archivos_global}[/green]")
print(f"  Tamaño total     : [yellow]{total_size_global / (1024**2):,.1f} MB[/yellow]")
print(f"  Beats totales    : [bold magenta]{total_beats_global:,}[/bold magenta]")
print(f"[bold white]{'─' * 50}[/bold white]")

# ── Distribución porcentual ─────────────────────────────────────────
if total_beats_global > 0:
    print(f"\n[bold cyan]Distribución de clases:[/bold cyan]")
    for r in resumen:
        if r["error"]:
            continue
        pct = (r["total_beats"] / total_beats_global) * 100
        barra = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
        print(f"  {r['label']:20s} {barra} {pct:5.1f}% ({r['total_beats']:,})")

print(f"\n[bold green]═══ VALIDACIÓN COMPLETADA ═══[/bold green]")
 

═══════════════════════════════════════════════

   VALIDACIÓN DE CAPA SILVER EN S3

═══════════════════════════════════════════════

Bucket : shazam-proyecto-ecg

Prefijo: silver/

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:725)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:443)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:443)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

##**Validación de capa Silver con las nuevas patologias**

In [0]:
# =====================================================================
# VALIDACIÓN DE CAPA SILVER EN S3 (AJUSTADO PARA SUB-SPLITS Y KEYS)
# Recorre todas las clases en silver/ e imprime:
#   - Cantidad de archivos por clase (.npz y .csv) explorando train/val/test
#   - Tamaño total por clase (MB)
#   - Total de muestras cargadas por clase (leyendo los .npz)
#   - Resumen general y dimensionalidad
# =====================================================================
import numpy as np
import io
from rich import print
from rich.table import Table

BUCKET_OUTPUT = 'shazam-proyecto-ecg'
S3_BASE = f"s3://{BUCKET_OUTPUT}/silver"

# Actualizado estrictamente con los S3_PREFIX de tus scripts de ingesta
CLASES = {
    "clase_0_normal":         "Normal (Latidos)",
    "clase_0_normal_500":      "Normal (Ventanas 5s)",
    "clase_0_normal_2000":      "Normal (Ventanas 8s)",
    "clase_1_afib_temporal":  "AFib (Ventanas 8s)",
    "clase_2_paced":          "Paced (Latidos)",
    "clase_2_sttc":           "sttc (Latidos)",
    "clase_2_sttc_beats":     "sttc (Ventanas 2s)",
    "clase_3_lbbb":           "LBBB (Latidos)",
}

SPLITS = ['train', 'val', 'test']

print("[bold cyan]═══════════════════════════════════════════════[/bold cyan]")
print("[bold cyan]   VALIDACIÓN DE CAPA SILVER EN S3[/bold cyan]")
print("[bold cyan]═══════════════════════════════════════════════[/bold cyan]")
print(f"[white]Bucket : {BUCKET_OUTPUT}[/white]")
print(f"[white]Prefijo: silver/[/white]\n")

# ── Recolectar info por clase ───────────────────────────────────────
resumen = []

for carpeta, label in CLASES.items():
    n_npz = 0
    n_csv = 0
    size_total = 0
    total_muestras = 0
    leads = 0
    muestras_length = 0
    n_archivos = 0
    
    # Iterar por los splits generados en la ingesta
    for split in SPLITS:
        ruta = f"{S3_BASE}/{carpeta}/{split}/"
        try:
            archivos = dbutils.fs.ls(ruta)
        except Exception:
            # Si el split no existe, simplemente continuamos
            continue
            
        for f in archivos:
            if f.isDir(): continue # Evitar leer directorios si los hubiera
            
            size_total += f.size
            n_archivos += 1
            nombre = f.name.lower()
            
            if nombre.endswith(".npz"):
                n_npz += 1
                try:
                    # Leer via Spark binaryFile
                    row = spark.read.format("binaryFile").load(f.path).first()
                    buf = io.BytesIO(row.content)
                    data = np.load(buf)
                    
                    # Identificar la llave correcta (Normal/AFib usan 'signals', Paced/LBBB usan 'x')
                    if 'signals' in data:
                        arr = data['signals']
                    elif 'x' in data:
                        arr = data['x']
                    else:
                        arr = data[list(data.keys())[0]]
                        
                    total_muestras += arr.shape[0]
                    
                    # Tomar la forma del primer array válido que encontremos
                    if leads == 0 and arr.ndim == 3:
                        leads = arr.shape[1]
                        muestras_length = arr.shape[2]
                        
                    data.close()
                    buf.close()
                except Exception as e:
                    print(f"[yellow]  ⚠ No se pudo leer {nombre}: {e}[/yellow]")
            elif nombre.endswith(".csv"):
                n_csv += 1

    # Agregar al resumen marcando error solo si no se encontró NADA en ningún split
    resumen.append({
        "clase": carpeta, 
        "label": label,
        "n_archivos": n_archivos, 
        "n_npz": n_npz, 
        "n_csv": n_csv,
        "size_bytes": size_total, 
        "total_muestras": total_muestras,
        "leads": leads, 
        "muestras_length": muestras_length, 
        "error": n_archivos == 0
    })

# ── Tabla de resultados ─────────────────────────────────────────────
tabla = Table(title="Resumen Capa Silver (Consolidado Train/Val/Test)", show_lines=True)
tabla.add_column("Clase S3", style="bold white")
tabla.add_column("Label / Tipo", style="cyan")
tabla.add_column("Archivos", justify="right")
tabla.add_column("NPZ", justify="right", style="green")
tabla.add_column("CSV", justify="right", style="green")
tabla.add_column("Tamaño (MB)", justify="right", style="yellow")
tabla.add_column("Total Muestras", justify="right", style="bold magenta")
tabla.add_column("Shape (C, L)", justify="center")

total_muestras_global = 0
total_size_global = 0
total_archivos_global = 0

for r in resumen:
    if r["error"]:
        tabla.add_row(
            r["clase"], r["label"],
            "[red]NO EXISTE/VACÍO[/red]", "-", "-", "-", "-", "-"
        )
        continue

    size_mb = r["size_bytes"] / (1024 * 1024)
    shape_str = f"({r['leads']}, {r['muestras_length']})" if r["leads"] > 0 else "-"

    tabla.add_row(
        r["clase"], r["label"],
        str(r["n_archivos"]),
        str(r["n_npz"]),
        str(r["n_csv"]),
        f"{size_mb:,.1f}",
        f"{r['total_muestras']:,}",
        shape_str,
    )

    total_muestras_global += r["total_muestras"]
    total_size_global += r["size_bytes"]
    total_archivos_global += r["n_archivos"]

print(tabla)

# ── Totales globales ────────────────────────────────────────────────
print(f"\n[bold white]{'─' * 60}[/bold white]")
print(f"[bold white]TOTALES GLOBALES DE INGESTA[/bold white]")
print(f"  Archivos totales : [green]{total_archivos_global}[/green]")
print(f"  Tamaño total     : [yellow]{total_size_global / (1024**2):,.1f} MB[/yellow]")
print(f"  Muestras totales : [bold magenta]{total_muestras_global:,}[/bold magenta] (Mixto: Latidos y Ventanas)")
print(f"[bold white]{'─' * 60}[/bold white]")

# ── Distribución porcentual ─────────────────────────────────────────
if total_muestras_global > 0:
    print(f"\n[bold cyan]Distribución de muestras por clase:[/bold cyan]")
    print("[dim italic]Nota: AFib reporta ventanas de 8s, el resto reporta latidos individuales.[/dim italic]")
    for r in resumen:
        if r["error"] or r["total_muestras"] == 0:
            continue
        pct = (r["total_muestras"] / total_muestras_global) * 100
        barra = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
        print(f"  {r['label']:20s} {barra} {pct:5.1f}% ({r['total_muestras']:,})")

print(f"\n[bold green]═══ VALIDACIÓN COMPLETADA ═══[/bold green]")

═══════════════════════════════════════════════

   VALIDACIÓN DE CAPA SILVER EN S3

═══════════════════════════════════════════════

Bucket : shazam-proyecto-ecg

Prefijo: silver/

...

Paced (Latidos)      ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   0.5% (3,277)

sttc (Latidos)       █████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  10.2% (69,249)

sttc (Ventanas 2s)   ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  12.8% (86,994)

LBBB (Latidos)       ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   1.1% (7,558)

═══ VALIDACIÓN COMPLETADA ═══

##**Validación de capa Silver con las nuevas patologias 12 derivaciones**

In [0]:
# =====================================================================
# VALIDACIÓN DE CAPA SILVER EN S3 (AJUSTADO PARA SUB-SPLITS Y KEYS)
# Recorre todas las clases en silver/ e imprime:
#   - Cantidad de archivos por clase (.npz y .csv) explorando train/val/test
#   - Tamaño total por clase (MB)
#   - Total de muestras cargadas por clase (leyendo los .npz)
#   - Resumen general y dimensionalidad
# =====================================================================
import numpy as np
import io
from rich import print
from rich.table import Table

BUCKET_OUTPUT = 'shazam-proyecto-ecg'
S3_BASE = f"s3://{BUCKET_OUTPUT}/silver_12leads"

# Actualizado estrictamente con los S3_PREFIX de tus scripts de ingesta
CLASES = {
    "clase_0_normal":         "Normal (Latidos)",
    "clase_0_normal_500":      "Normal (Ventanas 5s)",
    "clase_0_normal_2000":      "Normal (Ventanas 8s)",
    "clase_1_afib_temporal":  "AFib (Ventanas 8s)",
    "clase_2_sttc_beats":     "sttc (Ventanas 2s)",
    "clase_3_lbbb":           "LBBB (Latidos)",
    "clase_4_rbbb":           "RBBB (Latidos)",
    "clase_5_pvc":             "PVC (Latidos)",
 
}

SPLITS = ['train', 'val', 'test']

print("[bold cyan]═══════════════════════════════════════════════[/bold cyan]")
print("[bold cyan]   VALIDACIÓN DE CAPA SILVER EN S3[/bold cyan]")
print("[bold cyan]═══════════════════════════════════════════════[/bold cyan]")
print(f"[white]Bucket : {BUCKET_OUTPUT}[/white]")
print(f"[white]Prefijo: silver/[/white]\n")

# ── Recolectar info por clase ───────────────────────────────────────
resumen = []

for carpeta, label in CLASES.items():
    n_npz = 0
    n_csv = 0
    size_total = 0
    total_muestras = 0
    leads = 0
    muestras_length = 0
    n_archivos = 0
    
    # Iterar por los splits generados en la ingesta
    for split in SPLITS:
        ruta = f"{S3_BASE}/{carpeta}/{split}/"
        try:
            archivos = dbutils.fs.ls(ruta)
        except Exception:
            # Si el split no existe, simplemente continuamos
            continue
            
        for f in archivos:
            if f.isDir(): continue # Evitar leer directorios si los hubiera
            
            size_total += f.size
            n_archivos += 1
            nombre = f.name.lower()
            
            if nombre.endswith(".npz"):
                n_npz += 1
                try:
                    # Leer via Spark binaryFile
                    row = spark.read.format("binaryFile").load(f.path).first()
                    buf = io.BytesIO(row.content)
                    data = np.load(buf)
                    
                    # Identificar la llave correcta (Normal/AFib usan 'signals', Paced/LBBB usan 'x')
                    if 'signals' in data:
                        arr = data['signals']
                    elif 'x' in data:
                        arr = data['x']
                    else:
                        arr = data[list(data.keys())[0]]
                        
                    total_muestras += arr.shape[0]
                    
                    # Tomar la forma del primer array válido que encontremos
                    if leads == 0 and arr.ndim == 3:
                        leads = arr.shape[1]
                        muestras_length = arr.shape[2]
                        
                    data.close()
                    buf.close()
                except Exception as e:
                    print(f"[yellow]  ⚠ No se pudo leer {nombre}: {e}[/yellow]")
            elif nombre.endswith(".csv"):
                n_csv += 1

    # Agregar al resumen marcando error solo si no se encontró NADA en ningún split
    resumen.append({
        "clase": carpeta, 
        "label": label,
        "n_archivos": n_archivos, 
        "n_npz": n_npz, 
        "n_csv": n_csv,
        "size_bytes": size_total, 
        "total_muestras": total_muestras,
        "leads": leads, 
        "muestras_length": muestras_length, 
        "error": n_archivos == 0
    })

# ── Tabla de resultados ─────────────────────────────────────────────
tabla = Table(title="Resumen Capa Silver (Consolidado Train/Val/Test)", show_lines=True)
tabla.add_column("Clase S3", style="bold white")
tabla.add_column("Label / Tipo", style="cyan")
tabla.add_column("Archivos", justify="right")
tabla.add_column("NPZ", justify="right", style="green")
tabla.add_column("CSV", justify="right", style="green")
tabla.add_column("Tamaño (MB)", justify="right", style="yellow")
tabla.add_column("Total Muestras", justify="right", style="bold magenta")
tabla.add_column("Shape (C, L)", justify="center")

total_muestras_global = 0
total_size_global = 0
total_archivos_global = 0

for r in resumen:
    if r["error"]:
        tabla.add_row(
            r["clase"], r["label"],
            "[red]NO EXISTE/VACÍO[/red]", "-", "-", "-", "-", "-"
        )
        continue

    size_mb = r["size_bytes"] / (1024 * 1024)
    shape_str = f"({r['leads']}, {r['muestras_length']})" if r["leads"] > 0 else "-"

    tabla.add_row(
        r["clase"], r["label"],
        str(r["n_archivos"]),
        str(r["n_npz"]),
        str(r["n_csv"]),
        f"{size_mb:,.1f}",
        f"{r['total_muestras']:,}",
        shape_str,
    )

    total_muestras_global += r["total_muestras"]
    total_size_global += r["size_bytes"]
    total_archivos_global += r["n_archivos"]

print(tabla)

# ── Totales globales ────────────────────────────────────────────────
print(f"\n[bold white]{'─' * 60}[/bold white]")
print(f"[bold white]TOTALES GLOBALES DE INGESTA[/bold white]")
print(f"  Archivos totales : [green]{total_archivos_global}[/green]")
print(f"  Tamaño total     : [yellow]{total_size_global / (1024**2):,.1f} MB[/yellow]")
print(f"  Muestras totales : [bold magenta]{total_muestras_global:,}[/bold magenta] (Mixto: Latidos y Ventanas)")
print(f"[bold white]{'─' * 60}[/bold white]")

# ── Distribución porcentual ─────────────────────────────────────────
if total_muestras_global > 0:
    print(f"\n[bold cyan]Distribución de muestras por clase:[/bold cyan]")
    print("[dim italic]Nota: AFib reporta ventanas de 8s, el resto reporta latidos individuales.[/dim italic]")
    for r in resumen:
        if r["error"] or r["total_muestras"] == 0:
            continue
        pct = (r["total_muestras"] / total_muestras_global) * 100
        barra = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
        print(f"  {r['label']:20s} {barra} {pct:5.1f}% ({r['total_muestras']:,})")

print(f"\n[bold green]═══ VALIDACIÓN COMPLETADA ═══[/bold green]")

═══════════════════════════════════════════════

   VALIDACIÓN DE CAPA SILVER EN S3

═══════════════════════════════════════════════

Bucket : shazam-proyecto-ecg

Prefijo: silver/

...

sttc (Ventanas 2s)   ██████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  20.3% (86,603)

LBBB (Latidos)       ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   1.8% (7,656)

RBBB (Latidos)       ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   1.7% (7,134)

PVC (Latidos)        ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   4.7% (20,007)

═══ VALIDACIÓN COMPLETADA ═══